# 02. Feature Engineering
Dokumentasi encoding, scaling, dan justifikasi 6 fitur model XGBoost.

In [1]:
import pandas as pd
import numpy as np
import joblib

train = pd.read_csv("../data/train/train.csv")
val   = pd.read_csv("../data/val/val.csv")

FEATURES_CAT = ["day_type", "concert_size", "weather"]
FEATURES_NUM = ["concert_end_hour", "time_since_end_minutes", "distance_to_pickup_meters"]
print("Train shape:", train.shape)
train.head()

Train shape: (81468, 7)


,concert_end_hour,day_type,concert_size,weather,time_since_end_minutes,distance_to_pickup_meters,surge_multiplier
0,23,weekday,small,cloudy,2,380,1.87
1,23,weekend,large,cloudy,15,650,3.43
2,21,weekday,small,cloudy,44,650,1.05
3,24,weekday,large,cloudy,78,120,1.89
4,22,weekday,medium,clear,84,450,1.05


In [2]:
print("Encoding OrdinalEncoder:")
print("  day_type     : weekday=0, weekend=1")
print("  concert_size : small=0, medium=1, large=2")
print("  weather      : clear=0, cloudy=1, rain=2")
print()
print("Alasan OrdinalEncoder vs OneHotEncoder:")
print("- Ketiga fitur memiliki urutan logis")
print("- XGBoost memanfaatkan urutan langsung")
print("- Menghindari dimensionality explosion")

Encoding OrdinalEncoder:
  day_type     : weekday=0, weekend=1
  concert_size : small=0, medium=1, large=2
  weather      : clear=0, cloudy=1, rain=2

Alasan OrdinalEncoder vs OneHotEncoder:
- Ketiga fitur memiliki urutan logis
- XGBoost memanfaatkan urutan langsung
- Menghindari dimensionality explosion


In [3]:
encoder = joblib.load("../models/encoder.pkl")
scaler  = joblib.load("../models/scaler.pkl")
print("Statistik sebelum scaling:")
print(train[FEATURES_NUM].describe().round(2))
print("Mean:", scaler.mean_.round(2))
print("Std :", scaler.scale_.round(2))

Statistik sebelum scaling:
       concert_end_hour  time_since_end_minutes  distance_to_pickup_meters
count          81468.00                81468.00                   81468.00
mean              21.70                   43.45                     457.17
std                1.24                   25.63                     217.59
min               19.00                    0.00                     120.00
25%               21.00                   21.00                     380.00
50%               22.00                   43.00                     450.00
75%               23.00                   65.00                     650.00
max               24.00                   90.00                     900.00
Mean: [ 21.7   43.45 457.17]
Std : [  1.24  25.63 217.59]


In [4]:
justifikasi = {
    "concert_end_hour"         : "Jam larut = ojol langka = surge tinggi",
    "day_type"                 : "Weekend = demand lebih tinggi",
    "concert_size"             : "Kapasitas = volume penonton keluar bersamaan",
    "weather"                  : "Hujan = permintaan ojol melonjak",
    "time_since_end_minutes"   : "Decay surge seiring waktu",
    "distance_to_pickup_meters": "Makin jauh dari venue = surge lebih rendah",
}
for feat, reason in justifikasi.items():
    print(f"  {feat:<32} : {reason}")

  concert_end_hour                 : Jam larut = ojol langka = surge tinggi
  day_type                         : Weekend = demand lebih tinggi
  concert_size                     : Kapasitas = volume penonton keluar bersamaan
  weather                          : Hujan = permintaan ojol melonjak
  time_since_end_minutes           : Decay surge seiring waktu
  distance_to_pickup_meters        : Makin jauh dari venue = surge lebih rendah


## Kesimpulan
- OrdinalEncoder dipilih karena fitur kategorikal memiliki urutan logis
- StandardScaler memastikan fitur numerik pada skala yang sama
- Encoder dan scaler di-fit hanya pada train set (no data leakage)